# Intraday volume periodicity - illustration pedagogique CPU-only

> Navigation : [Source primaire](#source-primaire) | [Objectifs d'apprentissage](#objectifs-dapprentissage) | [Prerequis](#prerequis) | [Duree estimee](#duree-estimee) | [Construction du signal](#construction-du-signal-signal-synthetique) | [Spectre FFT](#spectre-de-puissance-fft) | [Lecture du resultat](#lecture-du-resultat) | [Limites](#limites-pedagogiques) | [Exercices etudiant](#exercices)

## Source primaire

Ce notebook illustre le concept spectral derriere l'article QuantConnect *Intraday Volume Periodicity* (#21066) sans pretendre implementer le rotator decrit par l'auteur. Il prend pour reference la source primaire du phenomene :

> Wu, L., Zhang, R. & Dai, Y. *Spectral Volume Models: Universal High-Frequency Periodicities in Intraday Trading Activities*. **Management Science** (nov. 2025), doi:10.1287/mnsc.2024.06215 ; preprint SSRN 4230610.

La these des auteurs est que le trading algorithmique produit des **periodicites persistantes** dans le flux de volume intraday, et que ces periodicites portent une prime de selection adverse exploitable cross-section. Ce notebook ne porte aucun claim de profitabilite : il illustre la **detection spectrale** sur une serie synthetique ou la periodicite est connue par construction, pour montrer ce que ressortirait d'un signal periodique faible dans une serie reelle.

**Pourquoi CPU-only**. Pas de backtest QC Cloud ici : la machine d'execution n'a pas les identifiants `QC_API_USER_ID` et la livraison est pedagogique, pas un rotator de production. Le notebook illustre un phenomene detectable par FFT sur une serie 1D synthetique, sans appel a un moteur QC.

**Verdict de l'issue #14991**. Suivant le triage de po-2024 (body de l'issue, acceptance deja cochee), le rotator lui-meme ne justifie ni projet ni notebook (signal proprietaire QC, in-sample, draft pending review, optimum au bord de grille). Ce notebook-ci est strictement l'illustration du concept spectral en amont : pedagogique, multi-cycles, sans pretention de rotator.

## Objectifs d'apprentissage

A l'issue de ce notebook, l'etudiant doit etre capable de :

1. Construire une serie temporelle intraday synthetique combinant une tendance lente (U-shape d'ouverture/close), une periodicite explicite et un bruit multiplicatif.
2. Calculer le spectre de puissance (FFT) d'une serie 1D et interpreter les pics comme des periodicites en unites metier (minutes).
3. Distinguer un pic de periodicite injectee du contenu basse frequence issu de l'enveloppe U-shape.
4. Nommer les precautions methodologiques qui rendent la detection spectrale robuste sur une serie reelle (fenetrage, detrending, agregation multi-jour).

## Prerequis

- Python 3.10+, `numpy >= 1.25`.
- Notions de transformee de Fourier discrete (spectre de puissance, frequence fondamentale, repli de spectre).
- Aucune connaissance finance avancee requise : U-shape et selection adverse sont definis inline.

## Duree estimee

Lecture + execution : 20 a 30 minutes. Exercices : 45 a 90 minutes supplementaires.

In [1]:
import numpy as np

rng = np.random.default_rng(seed=42)

# Fenetre U-shape typique d'une journee de trading US : pic a l'ouverture,
# creux le matin, pic au close, bruit multiplicatif. On y injecte une
# periodicite artificielle a 30 min (fond de panier du mid-day).
n_min = 390  # 6h30 = 390 minutes de trading US
t = np.arange(n_min)

u_shape = 1.0 + 1.2 * np.exp(-((t - 30) ** 2) / (2 * 25 ** 2)) \
              + 0.9 * np.exp(-((t - 380) ** 2) / (2 * 30 ** 2)) \
              + 0.3 * np.exp(-((t - 200) ** 2) / (2 * 60 ** 2))

periodicite = 0.25 * np.sin(2 * np.pi * t / 30)  # 30 minutes = 2 cycles/h
bruit = rng.lognormal(mean=0.0, sigma=0.6, size=n_min)
volume = u_shape * (1.0 + periodicite) * bruit

print(f"Serie synthetique generee : n={n_min} minutes, pic={volume.max():.1f},")
print(f"creux={volume.min():.1f}, SNR periodicite/bruit ~0.25/{np.std(bruit):.2f}")


Serie synthetique generee : n=390 minutes, pic=8.7,
creux=0.2, SNR periodicite/bruit ~0.25/0.76


## Spectre de puissance (FFT)

La transformee de Fourier discrete decompose un signal 1D en une somme de sinusoides. Le **spectre de puissance** est la moyenne du carre de l'amplitude par frequence : un pic dans ce spectre signale une sinusoidale dont la frequence (et donc la periode en minutes) porte une part importante de l'energie du signal.

Sur notre serie :

- En esperance, le pic principal (apres le continu DC filtre par `volume - volume.mean()`) tombe sur la frequence injectee a 30 min, soit `f = 1/30 = 0.0333` cycle/min (periode de 30 minutes).
- Le U-shape (gaussiennes a 30, 200 et 380 min) produit du contenu basse frequence (periodes longues, de l'ordre de la fenetre-journee entiere) qui peut eventuellement rivaliser avec le pic injecte selon le tirage du bruit (cf. *Limites* plus bas).

Implementation :

In [2]:
# Spectre de puissance (FFT). On attend un pic a f = 1/30 min^-1 = 0.033 cycle/min,
# soit une periode de 30 minutes. Le U-shape produit aussi du contenu basse
# frequence, distinct du pic periodique.
from numpy.fft import rfft, rfftfreq

spectrum = np.abs(rfft(volume - volume.mean())) ** 2
freqs = rfftfreq(n_min, d=1.0)  # 1 pas = 1 minute

idx_top = np.argsort(spectrum)[::-1][:5]
print("Top 5 frequences detectees :")
for i in idx_top:
    f = freqs[i]
    period_min = 1.0 / f if f > 0 else float('inf')
    print(f"  f = {f:.4f} cycle/min  ->  periode {period_min:.1f} min  "
          f"(puissance {spectrum[i]:.0f})")


Top 5 frequences detectees :
  f = 0.0333 cycle/min  ->  periode 30.0 min  (puissance 10234)
  f = 0.0051 cycle/min  ->  periode 195.0 min  (puissance 5542)
  f = 0.0026 cycle/min  ->  periode 390.0 min  (puissance 4571)
  f = 0.0641 cycle/min  ->  periode 15.6 min  (puissance 2340)
  f = 0.1821 cycle/min  ->  periode 5.5 min  (puissance 2046)


## Lecture du resultat

Sur la serie synthetique, le pic de puissance le plus eleve (apres le continu DC) doit tomber sur la periode injectee de 30 minutes. Cela confirme que la chaine de traitement restitue la periodicite injectee -- mais sur une serie ou la reponse est connue par construction, ce resultat ne prouve pas que le detecteur distingue une vraie periodicite du contenu basse frequence du U-shape et du bruit. C'est l'objet de la section suivante. Sur une serie reelle, la meme methode donne un pic dominant a la periodicite structurelle du marche (cf. Wu et al. 2025 - periodicites U-shape a 30 min et 60 min selon univers).

## Limites pedagogiques

Ce notebook n'aborde pas : (a) la stationnarite intraday (le U-shape produit du contenu basse frequence parasite), (b) le fenetrage de la FFT (Hann/Hamming), (c) la robustesse multi-jour (le spectre agrege), (d) la prime cross-section (selection adverse sur periodicite relative). Ce sont des grains distincts, hors du perimetre d'une illustration CPU-only.

L'article QC (#21066) et le papier M&S sous-jacent restent les references pour quiconque veut poursuivre ; ce notebook ne les remplace pas, il en montre un echo minimal.

## Controle negatif - le detecteur discrimine-t-il ?

La cellule precedente detecte la periode qu'elle a elle-meme injectee : c'est une
demonstration de bout en bout, pas une validation. Un test qui ne peut pas echouer ne
mesure rien. Pour que la detection ait un sens, il faut mesurer ce que l'outil rend
**en l'absence** de periodicite, puis comparer les deux regimes.

On repete donc l'experience sur deux bras, **20 seeds chacun** (meme construction que la
cellule 1, seule la sinusoide change) :

- **bras temoin** : U-shape + bruit log-normal, **aucune** periodicite injectee ;
- **bras periodique** : le meme signal augmente de la sinusoide 30 min d'amplitude 0.25.

Pour chaque serie on releve deux nombres, sans rien supposer sur leur distribution :

- le **rang** du bac de frequence a 30 min dans le spectre de puissance (1 = pic
  dominant, continu exclu) ;
- le **rapport de puissance** entre ce bac et la mediane du spectre.


In [3]:
# Controle negatif : deux bras, 20 seeds chacun. Seule difference avec la cellule 1 :
# inject=False supprime la sinusoide 30 min.
from numpy.fft import rfft, rfftfreq

SEEDS = [0, 1, 7, 42, 99, 2, 3, 5, 11, 13, 17, 23, 29, 31, 37, 41, 43, 47, 53, 61]
N_MIN = 390
AMP = 0.25


def build_volume(seed, inject, amplitude=AMP, n_min=N_MIN):
    """Serie synthetique intraday. inject=False => aucune periodicite (temoin)."""
    rng = np.random.default_rng(seed=seed)
    tt = np.arange(n_min)
    u = (1.0 + 1.2 * np.exp(-((tt - 30) ** 2) / (2 * 25 ** 2))
         + 0.9 * np.exp(-((tt - 380) ** 2) / (2 * 30 ** 2))
         + 0.3 * np.exp(-((tt - 200) ** 2) / (2 * 60 ** 2)))
    p = amplitude * np.sin(2 * np.pi * tt / 30) if inject else 0.0
    b = rng.lognormal(mean=0.0, sigma=0.6, size=n_min)
    return u * (1.0 + p) * b


def rank_and_ratio(vol):
    """Rang du bac a 30 min (1 = pic dominant) et rapport a la mediane du spectre."""
    n = len(vol)
    sp = np.abs(rfft(vol - vol.mean())) ** 2
    sp = sp.copy()
    sp[0] = 0.0                                   # on ecarte le continu (DC)
    i30 = int(np.argmin(np.abs(rfftfreq(n, d=1.0) - 1.0 / 30.0)))
    order = np.argsort(sp)[::-1]
    return int(np.where(order == i30)[0][0]) + 1, float(sp[i30] / np.median(sp[1:]))


temoin_ranks, temoin_ratio = [], []
period_ranks, period_ratio = [], []
for seed in SEEDS:
    r, q = rank_and_ratio(build_volume(seed, inject=False))
    temoin_ranks.append(r)
    temoin_ratio.append(q)
    r, q = rank_and_ratio(build_volume(seed, inject=True))
    period_ranks.append(r)
    period_ratio.append(q)

print(f"Bac 30 min -- {len(SEEDS)} seeds par bras, amplitude injectee = {AMP}")
print(f"{'bras':<11} {'pic dominant':>13} {'rang median':>12} {'rapport median':>15} "
      f"{'rapport min':>12} {'rapport max':>12}")
for nom, ranks, ratios in (("temoin", temoin_ranks, temoin_ratio),
                           ("periodique", period_ranks, period_ratio)):
    pic = sum(1 for r in ranks if r == 1)
    print(f"{nom:<11} {pic:>6}/{len(ranks):<6} {np.median(ranks):>12.0f} "
          f"{np.median(ratios):>15.1f} {min(ratios):>12.2f} {max(ratios):>12.2f}")

# Les deux distributions de rapport se recouvrent-elles ?
inversions = sum(1 for a in temoin_ratio for b in period_ratio if a >= b)
paires = len(temoin_ratio) * len(period_ratio)
print(f"\nPaires (temoin, periodique) ou le temoin fait aussi bien ou mieux : "
      f"{inversions} / {paires}")
print(f"Borne haute du temoin = {max(temoin_ratio):.2f} ; "
      f"borne basse du periodique = {min(period_ratio):.2f}")

i42 = SEEDS.index(42)
print(f"\nSeed 42 (la serie committee en cellule 1) : temoin -> rang "
      f"{temoin_ranks[i42]} (rapport {temoin_ratio[i42]:.2f}) ; "
      f"periodique -> rang {period_ranks[i42]} (rapport {period_ratio[i42]:.2f})")


Bac 30 min -- 20 seeds par bras, amplitude injectee = 0.25
bras         pic dominant  rang median  rapport median  rapport min  rapport max
temoin           0/20               44             2.0         0.20         5.25
periodique      11/20                1            19.3         5.32        39.34

Paires (temoin, periodique) ou le temoin fait aussi bien ou mieux : 0 / 400
Borne haute du temoin = 5.25 ; borne basse du periodique = 5.32

Seed 42 (la serie committee en cellule 1) : temoin -> rang 26 (rapport 3.46) ; periodique -> rang 1 (rapport 31.19)


### Seuil de detection

La comparaison precedente fixe l'amplitude a 0.25, celle du notebook. Reste la question
qui decide de l'utilite pratique : **a partir de quelle amplitude** la periodicite
devient-elle detectable ? On balaie l'amplitude injectee de 0 (temoin pur) a 0.50, sur
les 20 memes seeds, en comptant combien de fois le bac 30 min sort en tete.


In [4]:
# Seuil de detection : balayage d'amplitude, 20 seeds par point.
AMPLITUDES = [0.00, 0.02, 0.05, 0.10, 0.15, 0.25, 0.50]

print(f"{'amplitude':>10} {'bac 30 min en tete':>20} {'rang median':>12} "
      f"{'rapport median':>15}")
for amp in AMPLITUDES:
    ranks, ratios = [], []
    for seed in SEEDS:
        r, q = rank_and_ratio(build_volume(seed, inject=(amp > 0.0), amplitude=amp))
        ranks.append(r)
        ratios.append(q)
    pic = sum(1 for r in ranks if r == 1)
    print(f"{amp:>10.2f} {pic:>14}/{len(ranks):<5} {np.median(ranks):>12.0f} "
          f"{np.median(ratios):>15.1f}")


 amplitude   bac 30 min en tete  rang median  rapport median
      0.00              0/20              44             2.0
      0.02              0/20              48             1.9
      0.05              0/20              44             2.3
      0.10              0/20               8             5.3
      0.15              4/20               4             8.9
      0.25             11/20               1            19.3
      0.50             20/20               1            70.4


### Lecture du controle negatif

Les nombres ci-dessus sont mesures, pas illustratifs.

**Le detecteur discrimine.** Sur le rapport de puissance, les deux bras ne se recouvrent
pas : la borne haute du temoin vaut 5.25, la borne basse du bras periodique 5.32, soit
**0 inversion sur 400 paires** (temoin, periodique). Un rapport eleve signale donc bien
une periodicite reelle, et non le seul contenu basse frequence du U-shape.

**Mais le critere "pic dominant" n'est pas robuste.** A l'amplitude du notebook (0.25),
le bac 30 min n'est en tete que dans **11 series sur 20**. La serie committee (seed 42)
fait partie de ces 11 (rang 1, rapport 31.19, contre rang 26 et rapport 3.46 pour son
temoin) : la demonstration de la cellule 3 est donc un **cas favorable**, pas le
comportement generique du detecteur.

**Le seuil de detection est bas, mais reel.** Le balayage situe la premiere detection en
tete a une amplitude de **0.15** (4 series sur 20), et 20 sur 20 a 0.50. En dessous de
0.10 le pic injecte reste noye : 0 sur 20, rang median 8 a 0.10, et le rapport median
retombe au niveau du temoin (5.3 contre 2.0 au temoin pur).

**Ce que ce controle n'etablit pas.** Il porte sur une serie **synthetique**, a bruit
log-normal independant, et sur une periodicite **sinusoidale pure a 30 min**. Une serie
reelle a une periodicite qui derive, un bruit autocorrele et une non-stationnarite
intraday : ces nombres ne s'y transposent pas tels quels. Aucune conclusion de
rentabilite n'en est tiree ici.


## Exercices

### Exercice 1 - Injection de periodicite double (45 min)

**Objectif** : valider qu'on sait generer un signal a periodicites multiples et le distinguer en FFT.

**Consigne** : reproduire la cellule de generation du signal en ajoutant une periodicite a 60 min en plus de celle a 30 min (`periodicite2 = 0.15 * np.sin(2 * np.pi * t / 60)`), puis regenerer et relancer la FFT. Verifier que les deux periodicites apparaissent dans le top 5.

**Indice** : utilise `volume = u_shape * (1.0 + periodicite + periodicite2) * bruit`.

**Stub a completer** (stubs C.1 : pass autorise, ne pas lever d'erreur) :

In [5]:
# TODO etudiant : ajouter periodicite2 = 0.15 * np.sin(2 * np.pi * t / 60)
# puis recombiner dans volume, et relancer la FFT sur la nouvelle serie.
# Verifier que le top 5 contient maintenant a la fois 30 min et 60 min.
periodicite2 = 0.0  # placeholder - remplacer par la vraie expression
volume_double = u_shape * (1.0 + periodicite + periodicite2) * bruit

# Relancer la FFT sur volume_double (ne pas re-importer, deja importe plus haut)
spectrum_dbl = np.abs(rfft(volume_double - volume_double.mean())) ** 2
idx_top_dbl = np.argsort(spectrum_dbl)[::-1][:5]
print("Exercice a completer : remplacer periodicite2 par 0.15 * np.sin(2 * np.pi * t / 60)\n",
      "puis relancer la FFT pour voir apparaitre 30 min ET 60 min dans le top 5.\n",
      "Stub actuel : periodicite2 = 0.0, donc la serie est identique au cas de base.")

Exercice a completer : remplacer periodicite2 par 0.15 * np.sin(2 * np.pi * t / 60)
 puis relancer la FFT pour voir apparaitre 30 min ET 60 min dans le top 5.
 Stub actuel : periodicite2 = 0.0, donc la serie est identique au cas de base.


### Exercice 2 - Robustesse au seed (30 min)

**Objectif** : verifier la stabilite du pic 30 min sous 5 seeds distincts.

**Consigne** : modifier le `seed=42` en `seed=0`, `1`, `7`, `99`, regenerer et mesurer si la periodicite 30 min reste en tete (top 1) ou est devancee par une composante basse frequence du U-shape. Reporter le ratio seeds_ok / total_seeds.

**Indice** : encapsuler la generation + detection dans une fonction `one_run(seed) -> bool` (voir companion `intraday_volume_periodicity.py:detect_periods`).

**Stub a completer** :

In [6]:
# TODO etudiant : encapsuler dans one_run(seed) -> bool et compter les seeds
# qui placent la periodicite 30 min en tete de la FFT.
def one_run(seed):
    # Stub - a completer par l'etudiant (C.1 : pass autorise)
    pass

seeds = [0, 1, 7, 99]
results = [one_run(s) for s in seeds]
ok = sum(1 for r in results if r)
print(f"Exercice a completer : implementer one_run(seed) -> bool pour detecter \n",
      f"  si la periodicite 30 min est en tete de la FFT. Stub actuel : \n",
      f"  one_run retourne None, donc ok = 0/{len(seeds)}.")

Exercice a completer : implementer one_run(seed) -> bool pour detecter 
   si la periodicite 30 min est en tete de la FFT. Stub actuel : 
   one_run retourne None, donc ok = 0/4.


### Exemple guide - Fenetrage Hann (lecture 15 min, pas d'exercice)

**Objectif** : montrer la procedure complete de fenetrage avant FFT, pour que l'etudiant
puisse la reproduire dans son propre travail. Cet exemple est **resolu** : le code suit,
l'etudiant observe la sortie et la commente.

**Procedure** :
1. Construire la fenetre de Hann : `window = np.hanning(n_min)`.
2. Centrer le signal (meme operation que dans la cellule FFT principale) puis multiplier par la fenetre.
3. Calculer le spectre de puissance sur le signal fenetre.
4. Comparer au spectre non fenetre : la fuite spectrale autour du pic injecte a 30 min est reduite,
   au prix d'une legere perte de resolution (la fenetre etale legerement le pic).

**Sortie de reference** : le pic a 30 min reste en tete apres fenetrage, avec une puissance
absolue plus faible (la fenetre attenue les bords). Le contenu basse frequence du U-shape est
partiellement lisse par la fenetre. Voir le resultat juste en dessous.

In [7]:
# Exemple guide - fenetrage Hann, code resolu a executer et observer.
# La procedure est : Hann window -> centrage -> FFT -> top 5.
window = np.hanning(n_min)
volume_windowed = (volume - volume.mean()) * window

spectrum_w = np.abs(rfft(volume_windowed)) ** 2
idx_top_w = np.argsort(spectrum_w)[::-1][:5]
print("Top 5 apres fenetrage Hann (exemple guide) :")
for i in idx_top_w:
    f = freqs[i]
    period_min = 1.0 / f if f > 0 else float('inf')
    print(f"  f = {f:.4f} cycle/min  ->  periode {period_min:.1f} min  "
          f"(puissance {spectrum_w[i]:.0f})")

Top 5 apres fenetrage Hann (exemple guide) :
  f = 0.0333 cycle/min  ->  periode 30.0 min  (puissance 2736)
  f = 0.0000 cycle/min  ->  periode inf min  (puissance 1131)
  f = 0.0359 cycle/min  ->  periode 27.9 min  (puissance 1090)
  f = 0.0308 cycle/min  ->  periode 32.5 min  (puissance 1073)
  f = 0.2923 cycle/min  ->  periode 3.4 min  (puissance 731)


### Exercice 3 - Detection multi-jours (60 min)

**Objectif** : stabiliser la detection du pic 30 min par agregation spectrale sur N jours
independants, comme dans l'approche de Wu et al. 2025 (le papier agrege typiquement 5 a 20
journees pour amortir le bruit multiplicatif intra-journalier).

**Consigne** : generer `n_days = 10` series synthetiques (memes parametres U-shape et
periodicite, seeds differents), calculer le spectre de puissance de chaque serie, puis
faire la moyenne des spectres (`spectrum_mean = np.mean(np.array([...]), axis=0)`).
Verifier que le pic a 30 min devient plus prononce dans le spectre moyenne que dans
chacun des spectres individuels.

**Indice** : utiliser une boucle `for seed in range(n_days)` qui appelle
`generate_synthetic_volume(seed=seed)` (helper disponible dans le companion
`intraday_volume_periodicity.py`) ou recree localement la generation. La cle est de
regarder la **hauteur relative du pic 30 min vs le contenu basse frequence**, pas la
puissance absolue.

**Stub a completer** :

In [8]:
# TODO etudiant : generer n_days series, calculer chaque spectre, faire la moyenne,
# puis comparer le rapport pic 30 min / pic basse frequence entre spectre moyen et
# spectre individuel.
n_days = 10
spectra = []
for seed in range(n_days):
    # Stub : a completer par l'etudiant. Pour l'instant, on enregistre un seul
    # spectre (celui de la serie principale) pour que la cellule s'execute.
    _vol = locals().get('volume', None)
    if _vol is None:
        break
    _s = np.abs(rfft(_vol - _vol.mean())) ** 2
    spectra.append(_s)

if spectra:
    spectrum_mean = np.mean(np.array(spectra), axis=0)
    idx_top_m = np.argsort(spectrum_mean)[::-1][:5]
    print("Exercice a completer : generer 10 series avec seeds differents et\n",
          "  calculer la moyenne des spectres. Stub actuel : 1 seul spectre (la serie\n",
          "  de base), donc spectrum_mean == spectrum. La procedure est en place,\n",
          "  il manque la diversification des seeds.")
else:
    print("Exercice a completer : executer d'abord les cellules precedentes pour\n",
          "  que la variable 'volume' soit disponible.")

Exercice a completer : generer 10 series avec seeds differents et


   calculer la moyenne des spectres. Stub actuel : 1 seul spectre (la serie
   de base), donc spectrum_mean == spectrum. La procedure est en place,
   il manque la diversification des seeds.
